# Start

In [ ]:
# import libraries

# import pandas for manipulation
import pandas as pd

# mount google drive
from google.colab import drive

# Library needed to connect to the database
from sqlalchemy import create_engine, text
import psycopg2

# make timestamp
from datetime import datetime

# Authenticate and create a client
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# make timestamp
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

print('this runs qc output will all be timestamped: ' + timestamp)

this runs qc output will all be timestamped: 2025-10-29_20-54-03


In [ ]:
# Set up database pull via URL from the data team.
# (URL updated 2025-07-07 by Maisie to reflect new follower database location)
DATABASE_URL = 'postgres://readonly:p8973ecd68569dabd2e8d1bc7bd5446d35a0d457487e0a783c83de2ec8a1043cc@ec2-54-243-187-12.compute-1.amazonaws.com:5432/d3o895iig4poii'

if DATABASE_URL.startswith('postgres:'):
    DATABASE_URL = DATABASE_URL.replace('postgres:', 'postgresql:')

# Connect to the database
engine = create_engine(DATABASE_URL)
conn = engine.connect()

# create avaialable table list
def available_tables():
  # Show names of all tables
    query = text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
        """)
    table_names = pd.read_sql(query, conn)
    table_names = table_names['table_name'].tolist()
    return table_names

# function to convert sql tables to dataframes
def get_sql_table(table_name):
   df = pd.read_sql(table_name,conn)
   return df

# read and find the relevant tables for your work
available_tables()


['account_emailaddress',
 'account_emailconfirmation',
 'auth_group',
 'auth_group_permissions',
 'auth_permission',
 'auth_user',
 'auth_user_groups',
 'auth_user_user_permissions',
 'capacity_rating',
 'captive_industry_type',
 'captive_industry_use',
 'captive_non_industry_use',
 'ccs',
 'chp',
 'comment',
 'company',
 'company_owner',
 'country',
 'country_subdivision',
 'data_source',
 'django_admin_log',
 'django_content_type',
 'django_migrations',
 'django_session',
 'django_site',
 'entity_history',
 'entity_org_id',
 'entity_status',
 'entity_tag',
 'entity_type',
 'entity_update',
 'external_id_system',
 'fuel_category',
 'fuel_detail',
 'goget_project',
 'hydrogen_capable',
 'hydrogen_generating',
 'hydrogen_greenwashing',
 'installation_type',
 'iron_yearly_production',
 'language',
 'legal_entity_type',
 'lng_project',
 'lng_unit',
 'nuclear_model',
 'operator',
 'org_id',
 'plant',
 'plant_external_id',
 'plant_history',
 'plant_language',
 'plant_owner',
 'powerplant_un

In [ ]:
# define relevant table
table = 'goget_project' # change this for the table you want

In [ ]:
# filter status_timeline for only entries in goget_project
table_for_gsheet = get_sql_table(table)

# check table was properly gathered
display(table_for_gsheet.head())

,id,basinDatasource,concessionBlockDatasource,projectComplexDatasource,offshore,offshoreDatasource,productionType,productionTypeDatasource,fuelType,unitType,...,projectComplex_id,makeWIKI,namePrefix,concessionBlock2,concessionBlock3,concessionBlock4,concessionBlock5,concessionBlock6,concessionBlock7,capex
0,4145,[],[191985],[],onshore,[193518],conventional,[],oil,field,...,1.000003e+11,True,Palogue,None,None,None,None,None,None,NaN
1,1554,[],[],[],offshore,[193519],conventional,[193188],oil and gas,field,...,NaN,False,Newton-On-Trent,None,None,None,None,None,None,NaN
2,4663,[],[],[],onshore,[190576],conventional,[190576],gas,field,...,NaN,False,Shadoon,None,None,None,None,None,None,NaN
3,12694,[],"[193205, 193206]",[],onshore,[193518],None,[],oil and gas,field,...,NaN,True,San Roque Vaca Muerta,None,None,None,None,None,None,NaN
4,1872,[],[],[],onshore,[193518],None,[],gas,field,...,NaN,False,Otates,None,None,None,None,None,None,NaN


In [ ]:
# Create a new spreadsheet
spreadsheet_name = timestamp
try:
    sh = gc.open(spreadsheet_name)
except gspread.SpreadsheetNotFound:
    sh = gc.create(spreadsheet_name)
    print(f"Spreadsheet '{spreadsheet_name}' created.")

# Select the first worksheet
worksheet = sh.sheet1

# Clear existing data and update with the DataFrame
worksheet.clear()

# Convert the entire DataFrame to string type and fill NaN with empty strings before converting to list
data_to_upload = table_for_gsheet.astype(str).fillna('').values.tolist()

worksheet.update([table_for_gsheet.columns.values.tolist()] + data_to_upload)

print(f"DataFrame uploaded to spreadsheet '{spreadsheet_name}'.")

Spreadsheet '2025-10-29_20-54-03' created.
DataFrame uploaded to spreadsheet '2025-10-29_20-54-03'.
